# Phase 1: Data Foundation — Step 1.5: Synthesize Employee Skills

This notebook synthesizes the currently possessed skills for each employee in `employees.csv`. Since the raw files do not contain per-employee current skills, we construct them synthetically by mapping each employee's `JobRole` to the nearest O*NET occupation using `sentence-transformers` semantic similarity, retrieving the required skills for that occupation from `role_skills.csv`, and randomly sampling 40-70% of them (seeded by `EmployeeNumber` for reproducibility).

**IMPORTANT: Manual O*NET-SOC Code Overrides**
To ensure correct role mapping and avoid inappropriate skill assignments, we use direct O*NET-SOC code overrides for certain JobRoles. These overrides are applied at the code level for durability:
- Sales Executive → 41-9031.00 (Sales Engineers) - more appropriate for mid-level sales than Chief Executives or Sales Managers

In [1]:
import os
import random
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

proc_dir = os.path.join("data", "processed")
print(f"Working directory: {os.path.abspath(os.getcwd())}")

Working directory: C:\Users\Harshit Mishra\OneDrive\Desktop\enterprise_hr_ai


In [ ]:
unique_roles = list(df_emp["JobRole"].unique())
occ_titles = list(df_occ["Title"].unique())

# Manual O*NET-SOC code overrides for durable role mapping
# These are applied at the code level to ensure consistency across notebook re-runs
manual_code_override = {
    "Sales Executive": "41-9031.00"  # Sales Engineers - more appropriate for mid-level sales than Chief Executives (11-1011.00) or Sales Managers (11-2022.00)
}

# Fallback title-based overrides for other roles if needed
manual_title_override = {
    "Manufacturing Director": "Industrial Production Managers",
    "Research Director": "Natural Sciences Managers",
    "Sales Representative": "Sales Representatives of Services, Except Advertising, Insurance, Financial Services, and Travel",
    "Human Resources": "Human Resources Specialists",
    "Research Scientist": "Medical Scientists, Except Epidemiologists",
    "Manager": "General and Operations Managers"
}

print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Encoding titles...")
role_embeds = model.encode(unique_roles, convert_to_tensor=True)
occ_embeds = model.encode(occ_titles, convert_to_tensor=True)

similarities = cos_sim(role_embeds, occ_embeds)

# Create mapping dictionary
role_to_onet_title = {}
role_to_code = {}
threshold = 0.5

for i, role in enumerate(unique_roles):
    if role in manual_code_override:
        # Direct code override - skip semantic matching
        code = manual_code_override[role]
        title = df_occ[df_occ["O*NET-SOC Code"] == code]["Title"].iloc[0]
        role_to_onet_title[role] = title
        role_to_code[role] = code
        print(f"JobRole: '{role}' -> Manual Code Override: {code} ('{title}')")
    elif role in manual_title_override:
        # Title-based override (fallback)
        title = manual_title_override[role]
        code = title_to_code[title]
        role_to_onet_title[role] = title
        role_to_code[role] = code
        print(f"JobRole: '{role}' -> Manual Title Override: '{title}' (code: {code})")
    else:
        # Semantic matching
        sim_scores = similarities[i].tolist()
        best_idx = np.argmax(sim_scores)
        best_title = occ_titles[best_idx]
        best_score = sim_scores[best_idx]
        
        assert best_score >= threshold, f"No O*NET match found above threshold {threshold} for role '{role}'"
        role_to_onet_title[role] = best_title
        role_to_code[role] = title_to_code[best_title]
        print(f"JobRole: '{role}' -> Semantic Match: '{best_title}' (code: {role_to_code[role]}, score: {best_score:.4f})")

In [ ]:
# role_to_code is already built in the previous cell with manual overrides
# Just verify the mapping
for role, code in role_to_code.items():
    print(f"JobRole: '{role}' -> Code: {code}")

## 2. Match Job Roles to O*NET Titles using Embeddings
We encode the 9 unique JobRoles and all O*NET titles, calculate cosine similarities, and find the closest match. We use a manual override dictionary to correct known mismatches and avoid residual 'All Other' O*NET categories which lack detailed skills data.

In [ ]:
unique_roles = list(df_emp["JobRole"].unique())
occ_titles = list(df_occ["Title"].unique())

# Manual overrides for mismatches and All Other categories
manual_override = {
    "Manufacturing Director": "Industrial Production Managers",
    "Research Director": "Natural Sciences Managers",
    "Sales Representative": "Sales Representatives of Services, Except Advertising, Insurance, Financial Services, and Travel",
    "Human Resources": "Human Resources Specialists",
    "Research Scientist": "Medical Scientists, Except Epidemiologists",
    "Manager": "General and Operations Managers",
    "Sales Executive": "Sales Engineers"  # Better match for JobLevel 2-3 mid-level sales role than Chief Executives
}

print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Encoding titles...")
role_embeds = model.encode(unique_roles, convert_to_tensor=True)
occ_embeds = model.encode(occ_titles, convert_to_tensor=True)

similarities = cos_sim(role_embeds, occ_embeds)

# Create mapping dictionary
role_to_onet_title = {}
threshold = 0.5

for i, role in enumerate(unique_roles):
    if role in manual_override:
        role_to_onet_title[role] = manual_override[role]
        print(f"JobRole: '{role}' -> Manual Override: '{manual_override[role]}' (forced)")
    else:
        sim_scores = similarities[i].tolist()
        best_idx = np.argmax(sim_scores)
        best_title = occ_titles[best_idx]
        best_score = sim_scores[best_idx]
        
        assert best_score >= threshold, f"No O*NET match found above threshold {threshold} for role '{role}'"
        role_to_onet_title[role] = best_title
        print(f"JobRole: '{role}' -> Semantic Match: '{best_title}' (score: {best_score:.4f})")

## 3. Map Mapped Titles to O*NET-SOC Codes

In [4]:
title_to_code = dict(zip(df_occ["Title"], df_occ["O*NET-SOC Code"]))
role_to_code = {role: title_to_code[onet_title] for role, onet_title in role_to_onet_title.items()}
for role, code in role_to_code.items():
    print(f"JobRole: '{role}' -> Code: {code}")

JobRole: 'Sales Executive' -> Code: 11-2022.00
JobRole: 'Research Scientist' -> Code: 19-1042.00
JobRole: 'Laboratory Technician' -> Code: 29-2012.00
JobRole: 'Manufacturing Director' -> Code: 11-3051.00
JobRole: 'Healthcare Representative' -> Code: 29-2099.08
JobRole: 'Manager' -> Code: 11-1021.00
JobRole: 'Sales Representative' -> Code: 41-3091.00
JobRole: 'Research Director' -> Code: 11-9121.00
JobRole: 'Human Resources' -> Code: 13-1071.00


## 4. Synthesize Employee Current Skills
For each employee in the dataset, we identify their mapped `O*NET-SOC Code`, query the associated skills from `role_skills.csv`, and randomly sample between 40% and 70% of them (seeded by the employee's `EmployeeNumber` for perfect reproducibility).

In [5]:
employee_skills_data = []
skills_by_code = df_role_skills.groupby("O*NET-SOC Code")["Skill Name"].apply(list).to_dict()

for index, row in df_emp.iterrows():
    emp_id = row["EmployeeNumber"]
    job_role = row["JobRole"]
    code = role_to_code[job_role]
    
    available_skills = skills_by_code.get(code, [])
    if not available_skills:
        print(f"Warning: No skills found for code {code} (JobRole: {job_role})")
        continue
        
    # Seed based on EmployeeNumber to ensure reproducibility
    random.seed(int(emp_id))
    
    # Determine how many skills to sample (40-70%)
    num_skills = len(available_skills)
    sample_fraction = random.uniform(0.4, 0.7)
    num_to_sample = int(round(num_skills * sample_fraction))
    num_to_sample = max(1, min(num_to_sample, num_skills))
    
    sampled_skills = random.sample(available_skills, num_to_sample)
    
    for skill in sampled_skills:
        employee_skills_data.append({
            "employee_id": emp_id,
            "current_skill": skill,
            "source": "synthesized"
        })

df_emp_skills = pd.DataFrame(employee_skills_data)
print(f"Synthesized {len(df_emp_skills)} employee-skill records for {df_emp['EmployeeNumber'].nunique()} employees.")
print(df_emp_skills.head(10))

Synthesized 58417 employee-skill records for 1470 employees.
   employee_id         current_skill       source
0            1  Web browser software  synthesized
1            1     Splunk Enterprise  synthesized
2            1                   SAS  synthesized
3            1   Learning Strategies  synthesized
4            1           Google Docs  synthesized
5            1         Apple Keynote  synthesized
6            1  Microsoft SharePoint  synthesized
7            1        Yardi software  synthesized
8            1    Microsoft Exchange  synthesized
9            1  Microsoft PowerPoint  synthesized


## 5. Save Synthesized Employee Skills
We save the synthesized dataset to `data/processed/employee_skills.csv`.

**LOUD COMMENT / DOCSTRING**:
This dataset is **entirely synthetic** because the raw files do not contain information about what skills each individual employee currently possesses. To enable a functional skill-gap analysis and recommendation engine, we map employee `JobRole` to O*NET titles and sample a subset (40-70%) of their required skills. All records are marked with `source="synthesized"`.

In [6]:
# LOUD COMMENT: This dataset is entirely synthetic.
# In the raw source data, there is no mapping of individual employees to their possessed skills.
# To make the workforce intelligence and upskilling modules functional, this dataset maps each JobRole
# to its O*NET counterpart, queries its skills from role_skills.csv, and randomly samples 40-70% of them.
output_path = os.path.join(proc_dir, "employee_skills.csv")
df_emp_skills.to_csv(output_path, index=False)
print(f"Successfully saved synthesized skills to {output_path}")

Successfully saved synthesized skills to data\processed\employee_skills.csv
